In [1]:
import os 
os.chdir('/data/T9.data/import/P9/Xenium/')

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import commot as ct
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('WhRd',["#e5e5e5", "#fffacd", "red", "darkred"], N=256)  
import pySTIM as pst
import matplotlib.pyplot as plt

/root/miniconda3/envs/stlearn/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2026-08-07 10:57:41.951600: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
adataxe = sc.read_h5ad("combined_adata.h5ad")

In [4]:
adataxe

AnnData object with n_obs × n_vars = 679659 × 5000
    obs: 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'sampleid', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'log10GenesPerUMI', 'leiden', 'celltype', 'cell_type_RCTD', 'CN'
    obsm: 'X_pca', 'X_umap', 'commot-cellchat-sum-receiver', 'commot-cellchat-sum-sender', 'spatial'
    layers: 'counts'

In [5]:
adataxe.obs.index = adataxe.obs.index.str.replace(r'-S\d+$', '', regex=True)

In [6]:
#adataxe.obs = adataxe.obs.drop(columns=['cell_type_RCTD','sampleid'])

In [7]:
adataxe.obs = pd.DataFrame(index=adataxe.obs.index)

In [8]:
adataxe

AnnData object with n_obs × n_vars = 679659 × 5000
    obsm: 'X_pca', 'X_umap', 'commot-cellchat-sum-receiver', 'commot-cellchat-sum-sender', 'spatial'
    layers: 'counts'

In [9]:
select_cells = pd.read_csv('./STout/sub.anno.csv', index_col=0) #, skiprows=2

In [10]:

new_obs = pd.merge(
    adataxe.obs,          
    select_cells,           
    left_index=True,    
    right_index=True,  
    how='left'         
)


adataxe.obs = new_obs

In [11]:
print(adataxe.obs['sampleid'].value_counts())

sampleid
S1    192693
S7    127085
S6     90209
S2     81677
S3     67197
S4     54719
S5     36034
S8     30045
Name: count, dtype: int64


In [12]:
adataxe.obs['cell_type_RCTD'] = adataxe.obs['cell_type_RCTD'].astype('category')

In [13]:
adataxe.obs['sampleid'] = adataxe.obs['sampleid'].astype('category')

In [14]:
import matplotlib
matplotlib.use('Agg')  
pst.calculate_cn(adataxe, mode='radius', radius= 100 ,
                group="sampleid", cluster="cell_type_RCTD",
                plot=True,
                n_neighborhoods=12, figsize=(16,8))
# 保存
plt.savefig('./STout/Knn.radius.pdf', dpi=300, bbox_inches='tight')
plt.close()

In [15]:
print(adataxe.obs['CN'].value_counts())

CN
1     133221
10    109466
3      64606
7      56553
5      54457
2      51158
4      46377
11     45113
9      40690
6      35158
8      21578
0      21282
Name: count, dtype: int64


In [16]:
print(adataxe.obs['cell_type_RCTD'].value_counts())

cell_type_RCTD
Tumor                    83591
CAFs myCAF like s5       56473
CCR7 CD4T                47485
APOE Macro               34264
B Memory                 32733
Plasmablasts             32171
GZMK CD8T                28953
B Naive                  28065
unknown                  23160
ACKR1 Endo               22495
Treg                     22028
RGS5 Endo                19944
CAFs MSC iCAF-like s1    18385
FABP5 Macro              17624
PVL Differentiated s3    16010
CD1C cDC2                14545
Myoepithelial            13178
CAFs Transitioning s3    13092
CXCL12 Endo              12114
IL7R CD4T                10569
FDC                       9391
PVL Immature s1           9197
PVL_Immature s2           9040
Progenitors Epi           8749
SIGLEC1 Macro             8596
EGR1 Macro                8540
CXCL10 Macro              7824
IFNG CD8T                 6853
LAMP3 migDC               6635
CLEC9A cDC1               6624
IFIT1 Tcell               6169
Tfh                     

In [17]:
print(adataxe.obs['CN'].value_counts())

CN
1     133221
10    109466
3      64606
7      56553
5      54457
2      51158
4      46377
11     45113
9      40690
6      35158
8      21578
0      21282
Name: count, dtype: int64


In [18]:
df = pd.DataFrame({
    'cellid': adataxe.obs_names,
    'CN': adataxe.obs['CN']
})

In [19]:
df.to_csv('./CN.csv')

In [20]:
import matplotlib
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('WhRd',["#e5e5e5", "#fffacd", "red", "darkred"], N=256)  
import pySTIM as pst
adataxe.obs['cell_type_RCTD'] = adataxe.obs['cell_type_RCTD'].astype('category')
adataxe.obs['sampleid'] = adataxe.obs['sampleid'].astype('category')
adataxe.obs['cell_type_RCTD'] = adataxe.obs['cell_type_RCTD'].astype('category')

In [21]:
adata_xe = adataxe

In [22]:
import matplotlib.pyplot as plt
plt.get_cmap("Set3", lut = 12)
cmap_CN = plt.get_cmap("Set3", lut = 12)
hex_colors = [plt.cm.colors.rgb2hex(cmap_CN(i)) for i in range(cmap_CN.N)]
adata_xe.uns['CN_colors'] = hex_colors

sc.pl.spatial(
    adata_xe, color="CN", spot_size=50, frameon=False ,show=False
)

plt.tight_layout()
plt.savefig('./STout/plot_scatter.pdf', dpi=300, bbox_inches='tight')
plt.close()  # 

In [23]:
# adata.X = adata.raw.X
# adata.var_names_make_unique()
# sc.pp.normalize_total(adata, inplace=True)
# sc.pp.log1p(adata)

In [24]:
adata = adataxe

In [25]:
df_ligrec = ct.pp.ligand_receptor_database(database='CellChat', species='human', signaling_type=None)
print(df_ligrec.shape)
df_cellchat_filtered = ct.pp.filter_lr_database(df_ligrec, adata, min_cell_pct=0.01)
print(df_cellchat_filtered.shape)

(1939, 4)
(228, 4)


In [26]:
df_ligrec.columns = ['Ligand', 'Receptor', 'Pathway', 'Category']

In [27]:
def expand_interactions(row):
    ligands = row['Ligand'].split('_')
    receptors = row['Receptor'].split('_')
    interactions = [{'Ligand': ligand, 'Receptor': receptor, 'Pathway': row['Pathway'], 'Category': row['Category']}
                    for ligand in ligands for receptor in receptors]
    return pd.DataFrame(interactions)

expanded_df = pd.concat([expand_interactions(row) for _, row in df_ligrec.iterrows()], ignore_index=True)

In [28]:
expanded_df.to_csv("./STout/CellChat_db.csv") ## Both Visium and imputed Xenium are using same LR database.

In [29]:
## convert the visium coordinates to Xenium coordinates (um)
#adata.obsm['spatial'] = adata.obs[['x_align', 'y_align']].values

In [31]:
sampleid = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8']
ad_list = {}
for i in sampleid:
    print(i)
    ad = adata[adata.obs.sampleid == i].copy()
    ad_list[i] = ad

S1
S2
S3
S4
S5
S6
S7
S8


In [32]:
signal_sender = []
signal_receiver = []
for i in sampleid:
    signal_sender.append(ad_list[i].obsm['commot-cellchat-sum-sender'])
    signal_receiver.append(ad_list[i].obsm['commot-cellchat-sum-receiver'])

In [33]:
signal_sender_df = pd.concat(signal_sender)
signal_receiver_df = pd.concat(signal_receiver)

In [34]:
signal_sender_df = signal_sender_df.drop(columns = 's-total-total')
signal_receiver_df = signal_receiver_df.drop(columns = 'r-total-total')

In [35]:
signal_sender_df = signal_sender_df.reindex(adata.obs.index)

In [36]:
signal_receiver_df = signal_receiver_df.reindex(adata.obs.index)

In [37]:
adata.obsm['commot-cellchat-sum-sender'] = signal_sender_df
adata.obsm['commot-cellchat-sum-receiver'] = signal_receiver_df

In [38]:
counts_df = adata.obs.groupby(['CN','sampleid']).size().reset_index(name = 'counts')
valid = counts_df[counts_df['counts'] >=20]
keep_conditions = set(zip(valid['sampleid'], valid['CN']))
mask = adata.obs.apply(lambda row: (row['sampleid'], row['CN']) in keep_conditions, axis=1)

In [39]:
print(adata.shape)
adata_filtered = adata[mask]
print(adata_filtered.shape)

(679659, 5000)
(679637, 5000)


In [40]:
X_r = adata_filtered.obsm['commot-cellchat-sum-receiver'].values
colname_r = list(adata_filtered.obsm['commot-cellchat-sum-receiver'].columns.values )
X_s = adata_filtered.obsm['commot-cellchat-sum-sender'].values
colname_s = list(adata_filtered.obsm['commot-cellchat-sum-sender'].columns.values )
X_signal = np.concatenate( (X_r, X_s), axis=1 )
colname = colname_r + colname_s
import anndata
adata_signal = anndata.AnnData(X=X_signal, var=pd.DataFrame(index=colname))

In [41]:
adata_signal.obs["CN"] = np.array(adata_filtered.obs['CN'])
adata_signal.obs['sampleid'] = adata_filtered.obs['sampleid'].values

In [42]:
pathways = np.unique(df_ligrec.loc[:,'Pathway'])

In [43]:
pathway_adata = adata_signal[:,adata_signal.var_names.str.contains('|'.join(pathways))].copy()
pathway_adata = pathway_adata[:,pathway_adata.var_names.str.contains('r-')]
pathway_adata.obs['CN'] = pathway_adata.obs['CN'].astype('category')
sc.tl.rank_genes_groups(pathway_adata, 'CN', method='wilcoxon', key_added="rank_CN_pathway")
ranked_genes_df = sc.get.rank_genes_groups_df(pathway_adata, group=None, key="rank_CN_pathway")

In [44]:
cross_tab = pd.crosstab(adata.obs['cell_type_RCTD'], adata.obs['CN'])

### KNN CN5

In [45]:
CNN = list(range(0, 12))  
for i in CNN:
    print(i)
    bar_df = ranked_genes_df[(ranked_genes_df['group']== str(i) ) & (ranked_genes_df['scores']> 5  )].sort_values("logfoldchanges", ascending=False)
    bar_df['Pathway'] = bar_df['names'].str.split('-').str[1]
    bar_df = bar_df.head(20)
    
    fc = list(bar_df['logfoldchanges'])
    terms = list(bar_df['Pathway'])
    from matplotlib.ticker import MaxNLocator
    
    plt.figure(figsize=(5,8.5), dpi = 100)
        
    fig = sns.barplot(x=fc, y=terms, color="#E0B1CB", edgecolor=None, linewidth=0)
    fig.set_xlabel('log2FC', fontsize=12, labelpad=10)
    fig.set_ylabel(f'CN{i} Top Signaling Pathways', fontsize=12, labelpad=10)
    fig.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tick_params(axis='x', which='major', labelsize=10)
    fig.set_yticks([])
    fig.tick_params(axis='y', which='both', length=0)
    fig.yaxis.label.set_visible(True)
    
    for ii, annot in enumerate(terms):
            
        title_start= max(fig.axes.get_xlim())/200
        import textwrap
        lines = textwrap.wrap(annot, width=60)
        for line_number, line in enumerate(lines):
            fig.text(title_start, ii + line_number * 0.5, line, ha='left', va='center', fontsize=10)
    
    fig.spines['right'].set_visible(False)
    fig.spines['top'].set_visible(False)
    plt.savefig(f'./STout/CN{i}_signalingPathway.pdf', bbox_inches='tight', transparent=True, dpi = 300)
    plt.show() 
        

0
1
2
3
4
5
6
7
8
9
10
11


In [46]:
#Path = bar_df['Pathway']

In [47]:
pathways = np.unique(df_ligrec.loc[:,'Pathway'])

In [48]:
lr_adata = adata_signal[:,~adata_signal.var_names.str.contains('|'.join(pathways))].copy()
lr_adata = lr_adata[:,~lr_adata.var_names.str.contains('s-')].copy()

In [49]:
lr_adata.var_names = lr_adata.var_names.str.split('-', n=1).str[1]

In [50]:
top_lr = lr_adata.var.index.tolist()

In [52]:
top_lr = [lr for lr in top_lr if lr in lr_adata.var_names]
print(len(top_lr))

127


In [54]:
lr_adata.obs['CN'] = lr_adata.obs['CN'].astype('category')

In [55]:
lr_adata.obs['CN'] = 'CN' + lr_adata.obs['CN'].astype(str)

In [56]:
sc.tl.rank_genes_groups(lr_adata, 'CN', method='wilcoxon')

In [57]:
cn_order = [f'CN{i}' for i in range(12)]  # CN0, CN1, ..., CN11
lr_adata.obs['CN'] = pd.Categorical(lr_adata.obs['CN'], categories=cn_order, ordered=True)

In [60]:
adata

AnnData object with n_obs × n_vars = 679659 × 5000
    obs: 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'log10GenesPerUMI', 'n_counts', 'leiden', 'celltype', 'sampleid', 'Cluster', 'cnv_leiden', 'cnv_score', 'cnv_status', 'cell_type_RCTD', 'cell_id.1', 'CN'
    uns: 'CN_colors'
    obsm: 'X_pca', 'X_umap', 'commot-cellchat-sum-receiver', 'commot-cellchat-sum-sender', 'spatial'
    layers: 'counts'

In [61]:
adata.X = adata.layers["counts"].copy()

In [62]:
adata.raw = adata.copy()

In [63]:
print(adata.X[550:600, :5].toarray())  # 

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [64]:
adata.write_h5ad('./bbknn.add.cn.lr.h5ad')

In [65]:
r = adata.obsm['commot-cellchat-sum-receiver']

In [66]:
s = adata.obsm['commot-cellchat-sum-sender']

In [67]:
col_names = r.columns.tolist()

In [68]:
pd.DataFrame(col_names, columns=['column_names']).to_csv('column_names.csv', index=False)

In [69]:
print(adata.obs['cell_type_RCTD'].value_counts())

cell_type_RCTD
Tumor                    83591
CAFs myCAF like s5       56473
CCR7 CD4T                47485
APOE Macro               34264
B Memory                 32733
Plasmablasts             32171
GZMK CD8T                28953
B Naive                  28065
unknown                  23160
ACKR1 Endo               22495
Treg                     22028
RGS5 Endo                19944
CAFs MSC iCAF-like s1    18385
FABP5 Macro              17624
PVL Differentiated s3    16010
CD1C cDC2                14545
Myoepithelial            13178
CAFs Transitioning s3    13092
CXCL12 Endo              12114
IL7R CD4T                10569
FDC                       9391
PVL Immature s1           9197
PVL_Immature s2           9040
Progenitors Epi           8749
SIGLEC1 Macro             8596
EGR1 Macro                8540
CXCL10 Macro              7824
IFNG CD8T                 6853
LAMP3 migDC               6635
CLEC9A cDC1               6624
IFIT1 Tcell               6169
Tfh                     

In [73]:
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

ranked_genes_df['Pathway'] = ranked_genes_df['names'].str.split('-').str[1]


idx_max = ranked_genes_df.groupby('Pathway')['logfoldchanges'].idxmax()
deduped_df = ranked_genes_df.loc[idx_max].copy()
# ============================================================================


with PdfPages('./STout/all_CN_signalingPathways.pdf') as pdf:


  n_plots = len(range(12))
  n_cols = 6
  n_rows = (n_plots + n_cols - 1) // n_cols
  fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
  axes = axes.flatten()

  for i in range(12):
      ax = axes[i]

      bar_df = deduped_df[(deduped_df['group'] == str(i)) &
                          (deduped_df['scores'] > 2 )].sort_values(
          "logfoldchanges", ascending=False)

      if len(bar_df) == 0:
          ax.text(0.5, 0.5, f'No significant pathways for CN{i}',
                 ha='center', va='center', transform=ax.transAxes)
          ax.set_axis_off()
          continue

      bar_df = bar_df.head(15)

      fc = list(bar_df['logfoldchanges'])
      terms = list(bar_df['Pathway'])

      sns.barplot(x=fc, y=terms, color="#E0B1CB", edgecolor=None, linewidth=0, ax=ax)

      ax.set_xlabel('log2FC', fontsize=10)
      ax.set_ylabel(f'CN{i} Top Signaling Pathways', fontsize=10)
      ax.xaxis.set_major_locator(MaxNLocator(integer=True))
      ax.tick_params(axis='x', labelsize=8)
      ax.set_yticks([])
      ax.tick_params(axis='y', length=0)

      for ii, annot in enumerate(terms):
          title_start = max(ax.get_xlim()) / 200
          lines = textwrap.wrap(annot, width=60)
          for line_number, line in enumerate(lines):
              ax.text(title_start, ii + line_number * 0.5, line,
                     ha='left', va='center', fontsize=8)

      ax.spines['right'].set_visible(False)
      ax.spines['top'].set_visible(False)

  for i in range(n_plots, len(axes)):
      axes[i].set_axis_off()

  plt.tight_layout()
  pdf.savefig(fig, bbox_inches='tight')
  plt.close()


In [76]:
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap


sns.set_style("white") 
sns.set_context("notebook", font_scale=1.2)

colors_custom = ['#66B9B9', '#D7D7D7', '#E0B1CB']
custom_cmap = LinearSegmentedColormap.from_list('custom_cmap', colors_custom, N=256)


ranked_genes_df['Pathway'] = ranked_genes_df['names'].str.split('-').str[1]
idx_max = ranked_genes_df.groupby('Pathway')['logfoldchanges'].idxmax()
deduped_df = ranked_genes_df.loc[idx_max].copy()


with PdfPages('./STout/all_CN_signalingPathways.pdf') as pdf:
    
    n_plots = 12
    n_cols = 6
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
    axes = axes.flatten()
    
    for i in range(n_plots):
        ax = axes[i]
        

        bar_df = deduped_df[(deduped_df['group'] == str(i)) & 
                           (deduped_df['scores'] > 2)].sort_values(
                           "logfoldchanges", ascending=False)
        
        if len(bar_df) == 0:
            ax.text(0.5, 0.5, f'No significant pathways for CN{i}',
                   ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.set_axis_off()
            continue
        
        bar_df = bar_df.head(15)
        fc = list(bar_df['logfoldchanges'])
        terms = list(bar_df['Pathway'])

        ax.set_facecolor("white")
        ax.patch.set_facecolor("white")
        fig.patch.set_facecolor("white")
        

        max_abs_fc = max(abs(min(fc)), abs(max(fc))) if fc else 1
        norm_fc = [val / max_abs_fc for val in fc]  

        bar_colors = []
        for norm_val in norm_fc:

            color_idx = (norm_val + 1) / 2  
            bar_colors.append(custom_cmap(color_idx))
        
        bars = sns.barplot(x=fc, y=terms, palette=bar_colors, 
                          edgecolor='none', linewidth=0, ax=ax)
        

        for idx, (val, term) in enumerate(zip(fc, terms)):
            if val >= 0:
                x_pos = val + 0.02 * max_abs_fc
                ha = 'left'
            else:
                x_pos = val - 0.02 * max_abs_fc
                ha = 'right'
            ax.text(x_pos, idx, 
                   f'{val:.2f}', 
                   va='center', ha=ha, fontsize=7, color='black', alpha=0.7)
        

        ax.set_xlabel('log2 Fold Change', fontsize=11, fontweight='bold')
        ax.set_ylabel(f'CN{i} Top Signaling Pathways', fontsize=11, fontweight='bold')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.tick_params(axis='x', labelsize=9)
        

        ax.tick_params(axis='y', labelsize=8, length=0)
        ax.set_yticklabels(terms)  

        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        ax.spines['left'].set_visible(False)  #
        

        ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8, alpha=0.4)
    

    for i in range(n_plots, len(axes)):
        axes[i].set_axis_off()
    

    from matplotlib.patches import Patch
    
    plt.tight_layout(rect=[0, 0.03, 1, 1])  #
    pdf.savefig(fig, bbox_inches='tight', facecolor='white')  # 
    plt.close()

✅ PDF生成完成：./STout/all_CN_signalingPathways.pdf
